# Bronze Layer Ingestion Pipeline - Generic & Metadata-Driven

Generic notebook to ingest ANY source into Bronze Delta tables using LakeForge framework configuration.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_path", "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/configs/pipeline_config.json", "Config Path")
dbutils.widgets.text("source_system", "erp", "Source System")
dbutils.widgets.text("pipeline_name", "customers", "Pipeline (Source) Name")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load Type")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")

config_path = dbutils.widgets.get("config_path")
source_system = dbutils.widgets.get("source_system")
pipeline_name = dbutils.widgets.get("pipeline_name")
load_type = dbutils.widgets.get("load_type")
environment = dbutils.widgets.get("environment")

print(f"Executing Bronze Ingestion for: {pipeline_name} in {environment} ({load_type} mode)")

In [ ]:
# Imports & Path setup
import sys
sys.path.append("/Workspace/Users/jayarampogakula@gmail.com/lakeforge")

from dataclasses import asdict
from lakeforge import (
    ConfigParser,
    create_schema_drift_detector,
    create_dq_engine,
    create_bronze_writer,
    create_trust_engine
)

from pyspark.sql import functions as F
from datetime import datetime

print("✅ Framework libraries loaded")

In [ ]:
# Load Configurations
config = ConfigParser.parse_monolithic_pipeline_config(config_path)

ing_config = ConfigParser.get_ingestion_config(config, pipeline_name, environment)
dq_config = ConfigParser.get_dq_config(config, pipeline_name, environment)

target_table = f"{ing_config.target_catalog}.{ing_config.target_schema}.{ing_config.target_table}"
print(f"Target Table: {target_table}")

In [ ]:
# Load Source Data dynamically
df_raw = spark.read.format(ing_config.file_format) \
    .option("header", str(ing_config.header).lower()) \
    .option("inferSchema", str(ing_config.infer_schema).lower()) \
    .option("delimiter", ing_config.delimiter) \
    .load(ing_config.source_path)

print(f"✅ Raw data loaded: {df_raw.count()} rows")

In [ ]:
# Detect Schema Drift
drift_detector = create_schema_drift_detector(spark)
drift_results = drift_detector.detect_drift(
    source_df=df_raw,
    target_table=ing_config.target_table,
    catalog=ing_config.target_catalog,
    schema=ing_config.target_schema
)

if drift_results.get("has_drift"):
    print("⚠️ Schema drift detected:")
    print(drift_results)
else:
    print("✅ No schema drift detected")

In [ ]:
# Run Data Quality Engine validations
dq_engine = create_dq_engine(spark)
rules_list = [asdict(r) for r in dq_config.rules]

dq_results = dq_engine.validate_dataframe(
    df=df_raw,
    rules=rules_list,
    quarantine_failures=True
)

print(f"DQ Execution Summary: {dq_results['rules_passed']}/{dq_results['rules_executed']} rules passed")

In [ ]:
# Segregate and write clean / quarantine records
business_keys = config["sources"][pipeline_name]["business_key"]

if dq_results["quarantine_count"] > 0:
    # Left anti join to get clean rows
    clean_df = df_raw.join(dq_results["quarantine_df"], on=business_keys, how="left_anti")
else:
    clean_df = df_raw

# Write clean records
bronze_writer = create_bronze_writer(spark)
bronze_writer.write_to_bronze(
    df=clean_df,
    target_table=ing_config.target_table,
    catalog=ing_config.target_catalog,
    schema=ing_config.target_schema,
    mode=ing_config.mode,
    merge_keys=business_keys,
    add_audit_columns=True,
    source_system=source_system
)
print(f"✅ Wrote {clean_df.count()} clean rows to {target_table}")

# Write quarantine records if any
if dq_results["quarantine_count"] > 0:
    dq_engine.write_quarantine_table(
        quarantine_df=dq_results["quarantine_df"],
        catalog=dq_config.catalog,
        schema=dq_config.schema,
        table_name=dq_config.quarantine_table,
        validation_results=dq_results
)
    print(f"⚠️ Quarantined {dq_results['quarantine_count']} rows in {dq_config.quarantine_table}")

In [ ]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)
trust_score = trust_engine.calculate_trust_score(
    table_name=target_table,
    dq_results=dq_results,
    pipeline_stage="bronze",
    drift_results=drift_results
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")